# Union Patch Score Analysis for GT Relation Pairs

这个 notebook 使用仓库现有的 `VGDataset`、GT box、`edge_map` 路径约定、`crop_and_resize` 语义，以及本地可用的 CLIP ViT 权重，
对关系检测中的 subject-object **union 区域**计算并可视化 `Patch Score`。

- 默认样本来源：GT relation pairs
- 默认视觉编码器：CLIP `ViT-B/32`
- 默认输入模式：同时分析 `RGB` 和 `edge_map`
- 默认目标：单图 / 少量 pair 的可视化分析


## Cell 1：环境初始化

In [ ]:
import math
import os
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from IPython.display import display
from torchvision.transforms import functional as TF
import matplotlib.patches as patches

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from maskrcnn_benchmark.config import cfg
from maskrcnn_benchmark.data.datasets.visual_genome import VGDataset
from maskrcnn_benchmark.config.paths_catalog import DatasetCatalog
from maskrcnn_benchmark.modeling.roi_heads.relation_head.roi_relation_predictors import crop_and_resize
from CLIP import clip

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.style.use('default')
plt.rcParams['figure.figsize'] = (16, 8)
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10

print(f'Repo root: {REPO_ROOT}')
print(f'Using device: {device}')


## Cell 2：配置区

In [ ]:
# 可按需修改下面这些参数
config_file = ''  # 例如: 'configs/e2e_relation_X_101_32_8_FPN_1x.yaml'
config_opts = []  # 例如: ['MODEL.ROI_RELATION_HEAD.USE_EDGE_MAP', True]
dataset_name = None  # 例如: 'VG_stanford_filtered_test'；若为 None 则优先从 cfg 推断

split = 'test'
image_index = 0
pair_index = 0
image_mode = 'both'  # 可选: 'rgb', 'edge', 'both'
clip_model_name = 'ViT-B/16'  # 推荐先用 B/16，patch grid 更细
overlay_alpha = 0.4
show_boxes = True
topk_pairs = 5
heatmap_cmap = 'hot'

assert image_mode in {'rgb', 'edge', 'both'}

cfg_local = cfg.clone()
if config_file:
    cfg_local.merge_from_file(config_file)
if config_opts:
    cfg_local.merge_from_list(config_opts)
cfg_local.freeze()

print('Configured split:', split)
print('Configured image index:', image_index)
print('Configured pair index:', pair_index)
print('Configured image mode:', image_mode)
print('Configured CLIP model:', clip_model_name)
print('Configured dataset override:', dataset_name)


## Cell 3：数据集构建

In [ ]:
def _infer_dataset_name(cfg_obj, split_name, dataset_name_override=None):
    if dataset_name_override:
        return dataset_name_override

    split_attr = split_name.upper()
    dataset_names = getattr(cfg_obj.DATASETS, split_attr, ())
    if dataset_names:
        return dataset_names[0]

    return f'VG_stanford_filtered_{split_name}'


def build_dataset(cfg_obj, split_name, dataset_name_override=None):
    resolved_dataset_name = _infer_dataset_name(cfg_obj, split_name, dataset_name_override)
    dataset_info = DatasetCatalog.get(resolved_dataset_name, cfg_obj)
    if dataset_info['factory'] != 'VGDataset':
        raise ValueError(f'Expected VGDataset, got: {dataset_info["factory"]} for {resolved_dataset_name}')

    dataset_args = dict(dataset_info['args'])
    dataset_args.pop('capgraphs_file', None)
    dataset_args['transforms'] = None
    dataset = VGDataset(**dataset_args)
    return dataset, resolved_dataset_name


def _clone_target_to_eval_target(dataset, image_index):
    return dataset.get_groundtruth(image_index, evaluation=True, flip_img=False)


def load_sample(dataset, image_index):
    sample = dataset[image_index]
    if len(sample) == 3:
        image, _, image_id = sample
        edge_map = None
    elif len(sample) == 4:
        image, _, image_id, edge_map = sample
    else:
        raise ValueError(f'Unexpected sample size: {len(sample)}')

    target = _clone_target_to_eval_target(dataset, image_index)
    meta = {
        'image_id': image_id,
        'filename': dataset.filenames[image_index],
        'img_info': dataset.get_img_info(image_index),
        'ind_to_classes': dataset.ind_to_classes,
        'ind_to_predicates': dataset.ind_to_predicates,
        'has_edge_map': edge_map is not None,
    }
    return image, target, edge_map, meta


dataset, resolved_dataset_name = build_dataset(cfg_local, split, dataset_name_override=dataset_name)
image, target, edge_map, meta = load_sample(dataset, image_index)

print('Resolved dataset:', resolved_dataset_name)
print('Loaded image:', meta['filename'])
print('Image size:', image.size)
print('Has edge map:', meta['has_edge_map'])
print('Number of boxes:', len(target))
if target.has_field('relation_tuple'):
    print('Number of GT relations:', int(target.get_field('relation_tuple').shape[0]))
else:
    relation_map = target.get_field('relation')
    print('Number of GT relations:', int((relation_map > 0).sum().item()))


## Cell 4：GT relation pair 提取

In [ ]:
def extract_gt_pairs(target, ind_to_classes, ind_to_predicates):
    boxes = target.bbox.detach().cpu()
    labels = target.get_field('labels').detach().cpu()

    if target.has_field('relation_tuple'):
        relation_data = target.get_field('relation_tuple').detach().cpu()
        relation_triplets = relation_data.tolist()
    else:
        relation_map = target.get_field('relation').detach().cpu()
        relation_triplets = []
        nonzero = torch.nonzero(relation_map > 0, as_tuple=False)
        for sub_idx, obj_idx in nonzero.tolist():
            relation_triplets.append((sub_idx, obj_idx, int(relation_map[sub_idx, obj_idx].item())))

    pairs = []
    for idx, (sub_idx, obj_idx, rel_label) in enumerate(relation_triplets):
        sub_label = int(labels[sub_idx].item())
        obj_label = int(labels[obj_idx].item())
        sub_box = boxes[sub_idx].tolist()
        obj_box = boxes[obj_idx].tolist()
        pairs.append({
            'pair_id': idx,
            'sub_idx': int(sub_idx),
            'obj_idx': int(obj_idx),
            'rel_label': int(rel_label),
            'sub_name': ind_to_classes[sub_label],
            'predicate': ind_to_predicates[int(rel_label)],
            'obj_name': ind_to_classes[obj_label],
            'sub_box': [round(float(v), 2) for v in sub_box],
            'obj_box': [round(float(v), 2) for v in obj_box],
        })
    return pairs


def print_pair_table(pairs):
    if not pairs:
        print('当前样本没有 GT relation，停止后续分析。')
        return None
    table = pd.DataFrame(pairs)
    display(table)
    return table


gt_pairs = extract_gt_pairs(target, meta['ind_to_classes'], meta['ind_to_predicates'])
pair_table = print_pair_table(gt_pairs)

if not gt_pairs:
    raise RuntimeError('当前样本没有 GT relation，无法继续进行 union patch score 分析。')
if pair_index < 0 or pair_index >= len(gt_pairs):
    raise IndexError(f'pair_index={pair_index} 越界，可选范围: [0, {len(gt_pairs) - 1}]')

selected_pair = gt_pairs[pair_index]
print('Selected pair:')
print(selected_pair)


## Cell 5：Union crop 构建

In [ ]:
def _to_chw_tensor(image_tensor_or_pil):
    if isinstance(image_tensor_or_pil, Image.Image):
        return TF.to_tensor(image_tensor_or_pil)
    if torch.is_tensor(image_tensor_or_pil):
        tensor = image_tensor_or_pil.detach().cpu()
        if tensor.ndim == 2:
            tensor = tensor.unsqueeze(0)
        if tensor.ndim == 3:
            return tensor.float()
    raise TypeError(f'Unsupported image type: {type(image_tensor_or_pil)}')


def _compute_union_box(sub_box, obj_box):
    sub_box = torch.as_tensor(sub_box, dtype=torch.float32)
    obj_box = torch.as_tensor(obj_box, dtype=torch.float32)
    union_box = torch.cat((torch.min(sub_box[:2], obj_box[:2]), torch.max(sub_box[2:], obj_box[2:])), dim=0)
    return union_box


def _to_pil_image_3ch(tensor):
    tensor = tensor.detach().cpu().clamp(0.0, 1.0)
    if tensor.ndim == 4:
        tensor = tensor[0]
    if tensor.shape[0] == 1:
        tensor = tensor.repeat(3, 1, 1)
    return TF.to_pil_image(tensor)


def make_union_crop(image_tensor_or_pil, sub_box, obj_box, is_edge=False):
    image_tensor = _to_chw_tensor(image_tensor_or_pil)
    crop = crop_and_resize(image_tensor.unsqueeze(0), torch.tensor(sub_box), torch.tensor(obj_box)).squeeze(0)
    if is_edge and crop.shape[0] == 1:
        crop = crop.repeat(3, 1, 1)
    union_box = _compute_union_box(sub_box, obj_box)
    return {
        'tensor': crop,
        'pil': _to_pil_image_3ch(crop),
        'union_box': union_box,
        'sub_box': torch.tensor(sub_box, dtype=torch.float32),
        'obj_box': torch.tensor(obj_box, dtype=torch.float32),
    }


def _project_box_to_union(box, union_box, output_size=224):
    x1, y1, x2, y2 = [float(v) for v in box]
    ux1, uy1, ux2, uy2 = [float(v) for v in union_box]
    union_w = max(ux2 - ux1, 1e-6)
    union_h = max(uy2 - uy1, 1e-6)
    return [
        (x1 - ux1) / union_w * output_size,
        (y1 - uy1) / union_h * output_size,
        (x2 - ux1) / union_w * output_size,
        (y2 - uy1) / union_h * output_size,
    ]


rgb_union = make_union_crop(image, selected_pair['sub_box'], selected_pair['obj_box'], is_edge=False)
edge_union = None
if edge_map is not None:
    edge_union = make_union_crop(edge_map, selected_pair['sub_box'], selected_pair['obj_box'], is_edge=True)

print('Union box:', [round(float(v), 2) for v in rgb_union['union_box'].tolist()])
print('RGB crop size:', rgb_union['tensor'].shape)
if edge_union is None:
    print('Edge map unavailable for this sample.')
else:
    print('Edge crop size:', edge_union['tensor'].shape)


## Cell 6：CLIP token 提取

In [ ]:
try:
    clip_model, clip_preprocess = clip.load(clip_model_name, device=device)
    clip_model.eval()
    print(f'Loaded CLIP model: {clip_model_name}')
except Exception as exc:
    raise RuntimeError('本地 CLIP 权重不可用，请确认缓存权重存在并可被 clip.load(...) 读取。') from exc


def encode_clip_tokens(model, preprocess, pil_crop, device):
    image_input = preprocess(pil_crop).unsqueeze(0).to(device)
    with torch.no_grad():
        tokens = model.visual(image_input.type(model.dtype))
    return tokens.detach().float().cpu()


rgb_tokens = encode_clip_tokens(clip_model, clip_preprocess, rgb_union['pil'], device)
print('RGB token shape:', tuple(rgb_tokens.shape))
if edge_union is not None:
    edge_tokens = encode_clip_tokens(clip_model, clip_preprocess, edge_union['pil'], device)
    print('Edge token shape:', tuple(edge_tokens.shape))
else:
    edge_tokens = None


## Cell 7：Patch Score 计算

In [ ]:
def compute_patch_score(tokens):
    if tokens.ndim == 3:
        if tokens.shape[0] != 1:
            raise ValueError(f'Expected batch size 1, got {tokens.shape[0]}')
        tokens = tokens[0]

    cls_token = tokens[0].unsqueeze(0)
    patch_tokens = tokens[1:]
    num_patches = patch_tokens.shape[0]
    grid_size = int(math.sqrt(num_patches))
    if grid_size * grid_size != num_patches:
        raise ValueError(f'Patch token count {num_patches} cannot form a square grid')

    cls_token_expanded = cls_token.expand_as(patch_tokens)
    patch_scores = F.cosine_similarity(patch_tokens, cls_token_expanded, dim=-1)
    score_grid = patch_scores.view(grid_size, grid_size)
    score_map = F.interpolate(
        score_grid.unsqueeze(0).unsqueeze(0),
        size=(224, 224),
        mode='bilinear',
        align_corners=False,
    ).squeeze(0).squeeze(0)

    return {
        'cls_token': cls_token.squeeze(0),
        'patch_tokens': patch_tokens,
        'patch_scores': patch_scores,
        'score_grid': score_grid,
        'score_map': score_map,
        'grid_size': grid_size,
        'num_patches': num_patches,
    }


rgb_patch_output = compute_patch_score(rgb_tokens)
print('RGB patch grid shape:', tuple(rgb_patch_output['score_grid'].shape))
print('RGB raw score range:', float(rgb_patch_output['score_grid'].min()), float(rgb_patch_output['score_grid'].max()))

if edge_tokens is not None:
    edge_patch_output = compute_patch_score(edge_tokens)
    print('Edge patch grid shape:', tuple(edge_patch_output['score_grid'].shape))
    print('Edge raw score range:', float(edge_patch_output['score_grid'].min()), float(edge_patch_output['score_grid'].max()))
else:
    edge_patch_output = None


## Cell 8：可视化

In [ ]:
def render_patch_score_panel(ax_row, union_data, patch_output, title_prefix, overlay_alpha=0.4, show_boxes=True, heatmap_cmap='hot'):
    union_img = np.array(union_data['pil'])
    score_grid = patch_output['score_grid'].detach().cpu().numpy()
    score_map = patch_output['score_map'].detach().cpu().numpy()
    grid_size = patch_output['grid_size']
    local_sub_box = _project_box_to_union(union_data['sub_box'], union_data['union_box'])
    local_obj_box = _project_box_to_union(union_data['obj_box'], union_data['union_box'])

    ax_row[0].imshow(union_img)
    if show_boxes:
        sub_rect = patches.Rectangle(
            (local_sub_box[0], local_sub_box[1]),
            local_sub_box[2] - local_sub_box[0],
            local_sub_box[3] - local_sub_box[1],
            linewidth=2,
            edgecolor='lime',
            facecolor='none',
        )
        obj_rect = patches.Rectangle(
            (local_obj_box[0], local_obj_box[1]),
            local_obj_box[2] - local_obj_box[0],
            local_obj_box[3] - local_obj_box[1],
            linewidth=2,
            edgecolor='red',
            facecolor='none',
        )
        ax_row[0].add_patch(sub_rect)
        ax_row[0].add_patch(obj_rect)
    ax_row[0].set_title(f'{title_prefix} Union Crop')
    ax_row[0].axis('off')

    im = ax_row[1].imshow(score_grid, cmap=heatmap_cmap, interpolation='nearest')
    ax_row[1].set_title(
        f"{title_prefix} Patch Scores ({grid_size}x{grid_size})\n"
        f"Min: {score_grid.min():.3f}, Max: {score_grid.max():.3f}"
    )
    ax_row[1].set_xlabel('Patch X')
    ax_row[1].set_ylabel('Patch Y')
    plt.colorbar(im, ax=ax_row[1], fraction=0.046, pad=0.04)

    ax_row[2].imshow(union_img, alpha=0.6)
    im2 = ax_row[2].imshow(score_map, cmap=heatmap_cmap, alpha=overlay_alpha, interpolation='bilinear')
    if show_boxes:
        sub_rect2 = patches.Rectangle(
            (local_sub_box[0], local_sub_box[1]),
            local_sub_box[2] - local_sub_box[0],
            local_sub_box[3] - local_sub_box[1],
            linewidth=2,
            edgecolor='lime',
            facecolor='none',
        )
        obj_rect2 = patches.Rectangle(
            (local_obj_box[0], local_obj_box[1]),
            local_obj_box[2] - local_obj_box[0],
            local_obj_box[3] - local_obj_box[1],
            linewidth=2,
            edgecolor='cyan',
            facecolor='none',
        )
        ax_row[2].add_patch(sub_rect2)
        ax_row[2].add_patch(obj_rect2)
    ax_row[2].set_title(f'{title_prefix} Overlay')
    ax_row[2].axis('off')
    plt.colorbar(im2, ax=ax_row[2], fraction=0.046, pad=0.04)

    return {
        'raw_min': float(score_grid.min()),
        'raw_max': float(score_grid.max()),
        'grid_size': grid_size,
        'local_sub_box': local_sub_box,
        'local_obj_box': local_obj_box,
    }


def visualize_selected_pair(selected_pair, rgb_union, rgb_patch_output, edge_union=None, edge_patch_output=None, image_mode='both', overlay_alpha=0.4, show_boxes=True, heatmap_cmap='hot'):
    requested_modes = ['rgb', 'edge'] if image_mode == 'both' else [image_mode]
    valid_modes = []
    if 'rgb' in requested_modes:
        valid_modes.append('rgb')
    if 'edge' in requested_modes and edge_union is not None and edge_patch_output is not None:
        valid_modes.append('edge')

    if image_mode == 'edge' and not valid_modes:
        print('当前样本无 edge map，无法显示 edge 分支。')
        return None
    if image_mode == 'both' and edge_union is None:
        print('当前样本无 edge map，仅显示 RGB 分支。')

    fig, axes = plt.subplots(len(valid_modes), 3, figsize=(18, 5.5 * len(valid_modes)))
    if len(valid_modes) == 1:
        axes = np.expand_dims(axes, axis=0)

    stats = {}
    for row_idx, mode_name in enumerate(valid_modes):
        if mode_name == 'rgb':
            stats['rgb'] = render_patch_score_panel(
                axes[row_idx], rgb_union, rgb_patch_output, 'RGB', overlay_alpha=overlay_alpha, show_boxes=show_boxes, heatmap_cmap=heatmap_cmap
            )
        elif mode_name == 'edge':
            stats['edge'] = render_patch_score_panel(
                axes[row_idx], edge_union, edge_patch_output, 'Edge', overlay_alpha=overlay_alpha, show_boxes=show_boxes, heatmap_cmap=heatmap_cmap
            )

    union_box = [round(float(v), 2) for v in rgb_union['union_box'].tolist()]
    grid_info = rgb_patch_output['grid_size']
    fig.suptitle(
        f"image_index={image_index} | pair_index={selected_pair['pair_id']} | "
        f"{selected_pair['sub_name']} / {selected_pair['predicate']} / {selected_pair['obj_name']} | "
        f"union_box={union_box} | patch_grid={grid_info}x{grid_info}",
        fontsize=14,
        y=1.02,
    )
    plt.tight_layout()
    plt.show()
    return stats


_ = visualize_selected_pair(
    selected_pair=selected_pair,
    rgb_union=rgb_union,
    rgb_patch_output=rgb_patch_output,
    edge_union=edge_union,
    edge_patch_output=edge_patch_output,
    image_mode=image_mode,
    overlay_alpha=overlay_alpha,
    show_boxes=show_boxes,
    heatmap_cmap=heatmap_cmap,
)


## Cell 9：可选批量浏览

In [ ]:
def browse_topk_pairs(gt_pairs, image, edge_map, topk_pairs=5, image_mode='both', overlay_alpha=0.4, show_boxes=True, heatmap_cmap='hot'):
    total = min(topk_pairs, len(gt_pairs))
    print(f'Browsing first {total} GT pairs...')
    for idx in range(total):
        pair = gt_pairs[idx]
        rgb_union_local = make_union_crop(image, pair['sub_box'], pair['obj_box'], is_edge=False)
        rgb_tokens_local = encode_clip_tokens(clip_model, clip_preprocess, rgb_union_local['pil'], device)
        rgb_patch_output_local = compute_patch_score(rgb_tokens_local)

        edge_union_local = None
        edge_patch_output_local = None
        if edge_map is not None:
            edge_union_local = make_union_crop(edge_map, pair['sub_box'], pair['obj_box'], is_edge=True)
            edge_tokens_local = encode_clip_tokens(clip_model, clip_preprocess, edge_union_local['pil'], device)
            edge_patch_output_local = compute_patch_score(edge_tokens_local)

        _ = visualize_selected_pair(
            selected_pair=pair,
            rgb_union=rgb_union_local,
            rgb_patch_output=rgb_patch_output_local,
            edge_union=edge_union_local,
            edge_patch_output=edge_patch_output_local,
            image_mode=image_mode,
            overlay_alpha=overlay_alpha,
            show_boxes=show_boxes,
            heatmap_cmap=heatmap_cmap,
        )

# 取消下面注释即可顺序浏览当前图片前 topk_pairs 个 GT relations
# browse_topk_pairs(gt_pairs, image, edge_map, topk_pairs=topk_pairs, image_mode=image_mode, overlay_alpha=overlay_alpha, show_boxes=show_boxes, heatmap_cmap=heatmap_cmap)
